<a name="projet-7"></a>
# PROJET 7 : Loan Default Prediction #

<a name="contenu"></a>
## Contenu Partie N°1 ##
- Objectifs
- Import des données
- CSV en Parquet
- Etude des colonnes "loan status" et "issue_d"


<a name="objectifs"></a>
## Objectifs ##

_**Contexte :**_
- Les institutions financières doivent évaluer le risque de défaut de paiement des prêts.
- Des prédictions précises peuvent aider à prendre des décisions de prêt.
- Le Machine learning peut analyser l'historique des données afin de prédire les défauts.

_**Objectifs :**_
- Construire un modèle prédictif de défaut de paiement basé sur le profil des emprunteurs.
- Identifier les variables clés qui influent le plus sur le risque de défaut.
- Fournir des recommendations pour atténuer les risques.

_**Origine des données :**_
Le dataset utilisé provient de la plateforme Kaggle et regroupe l'historique complet des prêts acceptés par LendingClub entre 2007 et 2018. 
* **Volume :** Plus de 2,2 millions d'observations.
* **Richesse :** 151 variables décrivant le profil financier (revenus, dettes, historique de crédit) et les conditions du prêt (taux, mensualités).

- "All Lending Club loan data" (2007 through current Lending Club accepted and rejected loan data)
- URL : https://www.kaggle.com/datasets/wordsforthewise/lending-club/data

_**Qu'est-ce que LendingClub ?**_

**LendingClub** est une plateforme américaine de prêt entre particuliers (Peer-to-Peer Lending), fondée en 2006. Contrairement à une banque classique, elle agit comme un intermédiaire technologique :

* **Le concept :** Elle permet à des particuliers ou à des investisseurs institutionnels de prêter de l'argent directement à d'autres particuliers via une plateforme en ligne.

_**Défis :**_
- Traitement des classes déséquilibrées: gérer les jeux de données où les cas de défaut son minoritaires.
- Feature engineering: extraire les indicateurs les plus pertinents du profil des candidats.
- Interprétabilité du modèle: transformer les résultats techniques enleviers d'action concrets.


In [ ]:
import gc
# import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
# import os
import pandas as pd
# import plotly.express as px
# import scipy.stats as sps
import seaborn as sns

<a name="import"></a>
## Import des données ##


In [ ]:
df = pd.read_csv("DATA/accepted_2007_to_2018Q4.csv", engine='pyarrow')

# df = pd.read_csv("DATA/accepted_2007_to_2018Q4.csv",low_memory=False) 
# "low_memory=False" nécessite un temps de traitement bien supérieur.

In [ ]:
# Correction du format de 'id'

df['id'] = df['id'].astype(str)

In [ ]:
# Conversion de la colonne issue_d en format datetime

df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')

<a name="csv-en-parquet"></a>
## CSV en Parquet ## 


Le passage au format **Apache Parquet** est une étape importante pour ce projet :
* **Vitesse :** Lecture jusqu'à 10x plus rapide qu'un CSV.
* **Mémoire :** Compression drastique des données (passage de Go à Mo).
* **Fiabilité :** Conservation native des types de colonnes (dates, nombres), évitant les erreurs de typage lors des chargements futurs.

In [ ]:
# Sauvegarder en parquet

df.to_parquet("DATA/accepted_2007_to_2018Q4.parquet")

print("Conversion terminée !")

In [ ]:
# Nettoyer la mémoire vive (RAM)

# Suppression du DataFrame chargé depuis le CSV
if 'df' in locals():
    del df

# Libération de la mémoire
gc.collect()

# Chargement du fichier Parquet (beaucoup plus léger)
df = pd.read_parquet("DATA/accepted_2007_to_2018Q4.parquet")

print("Données chargées depuis le format Parquet !")
print(f"Dimensions du dataset : {df.shape}")

Notre base de données contient 2260701 lignes et 151 colonnes 

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
## Déterminons le nombre de valeurs manquantes par colonnes
df.isnull().sum()

In [ ]:
## pourcentage de valeurs manquantes par colonne
(df.isnull().mean()*100)

On remarque qu'il y'a des variables qui ont de grands nombres de valeurs manquantes.

In [ ]:
## les variables ayant plus de 80% de valeurs manquantes
missing_values_percentage= df.isna().mean()[df.isna().mean() >= 0.8]

In [ ]:
missing_values_percentage

On remarque qu'il y'a des colonnes qui n'ont pratiquement pas de valeurs.

In [ ]:
missing_values_percentage.count()

Il y a 39 variables qui ont plus de 80% de valeurs manquantes.


In [ ]:
## L'état des prêts.
stats_pret = pd.concat([df['loan_status'].value_counts(), 
                df['loan_status'].value_counts(normalize=True) * 100], 
               axis=1, keys=['Nombre', 'Pourcentage (%)'])

print(stats_pret)

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(y='loan_status', data=df)
plt.title("Distribution de loan_status")
plt.show()

Nous avons un bon apperçu des prêts déjà remboursés, de ceux en cours (qui n'ont fait l'objet d'aucun retard) ainsi que la part des prêts qui ont fait défaut.

In [ ]:
num_vars = df.select_dtypes(include=['int64','float64']).columns
cat_vars = df.select_dtypes(include=['object', 'str']).columns

print("Variables numériques :", len(num_vars))
print("Variables catégorielles :", len(cat_vars))


La base contient 113 variables numériques et 37 variables catégorielles.

In [ ]:
for col in ['loan_amnt','annual_inc','dti']:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()

<a name="etude-loan-status-et-issue-d"></a>
## Etude des colonnes "loan status" et "issue_d" ## 


In [ ]:
df['loan_status'].value_counts()

In [ ]:
# Filtre pour ne garder que les lignes "Current"
current_loans = df[df['loan_status'] == 'Current']

# Calcule des occurrences de issue_d sur ce sous-ensemble
counts = current_loans['issue_d'].value_counts().sort_index(ascending=False)

print("Aperçu des occurrences :")
print(counts)

In [ ]:
# Séparation en deux groupes en utilisant une simple comparaison de chaînes de caractères
mask_recent = counts.index >= '2017-01'
mask_old = counts.index <= '2016-12'

# Calcule de la somme pour chaque groupe
count_2017_2018 = counts[mask_recent].sum()
count_2016_and_before = counts[mask_old].sum()

print(f"Occurrences de 'Current' (2017-01 à 2018-12) : {count_2017_2018}")
print(f"Occurrences de 'Current' (2016-12 et inférieur) : {count_2016_and_before}")

In [ ]:
# Retrait des "current" (récents comme anciens)
lignes_a_supprimer = (df['loan_status'] == 'Current') # & (df['issue_d'] >= '2017-01-01')

# On met à jour le dataframe
df = df[~lignes_a_supprimer].copy()

# Vérification
print(f"Nombre de lignes restantes dans le dataframe après nettoyage : {len(df)}")

In [ ]:
# Sauvegardes en vue de la partie suivante

# Données

df.to_parquet("DATA/accepted_2007_to_2018Q4_filtered.parquet")
print("Fichier préparé pour l'étape suivante.")

In [ ]:
# 1. Liste des variables et DataFrames à supprimer de la Partie A
a_purger = [
    'df', 'missing_values_percentage', 'stats_pret', 'num_vars', 'cat_vars',
    'current_loans', 'counts', 'mask_recent', 'mask_old', 
    'count_2017_2018', 'count_2016_and_before', 'lignes_a_supprimer'
]

# 2. Suppression de l'environnement 
for var in a_purger:
    if var in locals():
        del locals()[var]

# 3. Libération de la mémoire
gc.collect()

print("🧹 Mémoire purgée avec succès ! Cette partie est terminée et le notebook est prêt à être fermé.")

----
